# Lab Assignment 3: Time Series Analysis

**Name:** Parimal Ahire  
**PRN:** 202301040067  
**Course:** Deep Learning Lab  
**GitHub Repository:** https://github.com/ParimalAhire/Lab-Assignment-3-TimeSeries


## Aim
To perform Time Series Analysis using:
1. **Time Series Properties Analysis** — Trend, Seasonality, Stationarity, ACF/PACF
2. **Hidden Markov Model (HMM)** — Forward Algorithm, Viterbi Algorithm, Future State Prediction
3. **ARIMA** — Stationarity check, model fitting, forecasting

Dataset: **Apple Stock Price (AAPL)** fetched using `yfinance`


## Algorithm
1. Import all libraries once
2. Load AAPL stock price dataset (shared across all parts)
3. **Part 1:** Time Series Properties Analysis
4. **Part 2:** Hidden Markov Model (HMM) — manually implemented
5. **Part 3:** ARIMA — fit and forecast
6. Compare HMM vs ARIMA predictions


---
## Step 1: Import All Libraries
> All imports done **once** and shared across all parts.


In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Time Series
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

# Stock data
import yfinance as yf

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

print('All libraries imported successfully!')


## Step 2: Load Dataset
> Dataset loaded **once** and shared across all three parts.

**Dataset:** Apple Inc. (AAPL) Daily Closing Stock Price  
**Period:** 2015 – 2023  
**Source:** Yahoo Finance via `yfinance`


In [ ]:
# Download AAPL stock data
ticker = yf.Ticker('AAPL')
df_raw = ticker.history(start='2015-01-01', end='2023-01-01')

# Use only Close price
df = df_raw[['Close']].copy()
df.columns = ['Price']
df.index = pd.to_datetime(df.index.date)  # clean date index

print('Dataset loaded!')
print(f'Shape        : {df.shape}')
print(f'Date range   : {df.index[0]} to {df.index[-1]}')
print(f'Total rows   : {len(df)}')
print(f'Missing vals : {df.isnull().sum().values[0]}')
print('\nFirst 5 rows:')
print(df.head())
print('\nLast 5 rows:')
print(df.tail())


In [ ]:
# Plot raw stock price
plt.figure(figsize=(12, 4))
plt.plot(df['Price'], color='steelblue', linewidth=1.5)
plt.title('AAPL Stock Price (2015–2023)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'\nMin Price: ${df["Price"].min():.2f}')
print(f'Max Price: ${df["Price"].max():.2f}')
print(f'Mean Price: ${df["Price"].mean():.2f}')


---
# Part 1: Time Series Properties Analysis
---

## Step 3: Trend & Rolling Statistics

**Trend** — long-term direction of the series  
**Rolling Mean** — smoothed average over a window  
**Rolling Std** — how much variation exists over time


In [ ]:
# Rolling mean and std
rolling_mean = df['Price'].rolling(window=30).mean()
rolling_std  = df['Price'].rolling(window=30).std()

plt.figure(figsize=(12, 5))
plt.plot(df['Price'],    color='steelblue', linewidth=1,   label='Original',     alpha=0.7)
plt.plot(rolling_mean,   color='red',       linewidth=2,   label='Rolling Mean (30d)')
plt.plot(rolling_std,    color='green',     linewidth=1.5, label='Rolling Std (30d)')
plt.title('AAPL — Trend, Rolling Mean & Rolling Std', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Observation: Rising rolling mean confirms an UPWARD TREND over 2015–2023')


## Step 4: Seasonality & Decomposition

**Seasonal Decomposition** breaks the series into:
- **Trend** component
- **Seasonal** component (repeating patterns)
- **Residual** component (noise)


In [ ]:
# Resample to monthly for cleaner decomposition
df_monthly = df['Price'].resample('M').mean()

# Seasonal decomposition
decomposition = seasonal_decompose(df_monthly, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))

axes[0].plot(decomposition.observed,  color='steelblue'); axes[0].set_title('Observed');  axes[0].grid(True, alpha=0.3)
axes[1].plot(decomposition.trend,     color='red');       axes[1].set_title('Trend');     axes[1].grid(True, alpha=0.3)
axes[2].plot(decomposition.seasonal,  color='green');     axes[2].set_title('Seasonal');  axes[2].grid(True, alpha=0.3)
axes[3].plot(decomposition.resid,     color='orange');    axes[3].set_title('Residual');  axes[3].grid(True, alpha=0.3)

plt.suptitle('Seasonal Decomposition — AAPL Monthly Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 5: Stationarity Check (ADF Test)

A time series is **stationary** if its mean and variance are constant over time.  
**ADF Test (Augmented Dickey-Fuller):**
- p-value < 0.05 → **Stationary** ✅
- p-value > 0.05 → **Non-Stationary** ❌ → apply differencing


In [ ]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    print(f'=== ADF Test: {name} ===')
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'p-value       : {result[1]:.4f}')
    print(f'Critical Values:')
    for key, val in result[4].items():
        print(f'   {key}: {val:.4f}')
    if result[1] <= 0.05:
        print('Result: STATIONARY ✅ (p <= 0.05)')
    else:
        print('Result: NON-STATIONARY ❌ (p > 0.05) — differencing needed')
    print()

# Test original series
adf_test(df['Price'], 'Original Price')

# Apply first differencing
df['Price_diff'] = df['Price'].diff()

# Test differenced series
adf_test(df['Price_diff'], 'First Differenced Price')


In [ ]:
# Plot original vs differenced
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(df['Price'],      color='steelblue', linewidth=1)
axes[0].set_title('Original Price (Non-Stationary)')
axes[0].set_ylabel('Price (USD)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['Price_diff'], color='tomato',    linewidth=1)
axes[1].set_title('First Differenced Price (Stationary)')
axes[1].set_ylabel('Price Change')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Stationarity: Original vs Differenced', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 6: ACF & PACF Plots

**ACF (AutoCorrelation Function)** → helps determine **q** (MA order) for ARIMA  
**PACF (Partial AutoCorrelation Function)** → helps determine **p** (AR order) for ARIMA


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df['Price_diff'].dropna(),  ax=axes[0], lags=40, title='ACF — Differenced Price')
plot_pacf(df['Price_diff'].dropna(), ax=axes[1], lags=40, title='PACF — Differenced Price')

axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)

plt.suptitle('ACF & PACF (Used to determine ARIMA p, q values)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('ACF  → significant lags help choose q (MA order)')
print('PACF → significant lags help choose p (AR order)')
print('d = 1 (one differencing made series stationary)')
print('\nSelected: ARIMA(p=1, d=1, q=1)')


---
# Part 2: Hidden Markov Model (HMM)
---

## Overview
HMM is used to find **hidden states** behind observed data.

For stock prices we define:
- **Hidden States:** Bull 📈 (rising), Bear 📉 (falling), Neutral ➡️ (stable)
- **Observations:** User input sequence (U = Up, D = Down, S = Stable)

**Two algorithms:**
1. **Forward Algorithm** — probability of the observed sequence
2. **Viterbi Algorithm** — most likely sequence of hidden states


## Step 7: User Input — Observation Sequence

In [ ]:
# Observation mapping
obs_full = {'U': 'Up', 'D': 'Down', 'S': 'Stable'}
obs_map  = {'Up': 0, 'Down': 1, 'Stable': 2}

print('Enter stock price observations:')
print('  U = Up   | D = Down   | S = Stable')
print('  Example: U D U S D')

choice = input('\nChoose (1: Manual input, 2: Random): ')

if choice == '1':
    user_input = input('Enter observations (e.g. U D U S D): ')
    short_obs  = user_input.strip().split()
    obs_seq = []
    for o in short_obs:
        o = o.upper()
        if o not in obs_full:
            print(f'Invalid input: {o}. Use U, D or S only.')
            raise ValueError('Invalid observation')
        obs_seq.append(obs_full[o])
else:
    obs_seq = [np.random.choice(['Up', 'Down', 'Stable']) for _ in range(10)]

print(f'\nObservation Sequence ({len(obs_seq)} steps):')
print(' → '.join(obs_seq))


## Step 8: Define HMM Parameters

**Three core components of HMM:**

| Component | Symbol | Meaning |
|---|---|---|
| Transition Matrix | A | P(next state \| current state) |
| Emission Matrix | B | P(observation \| state) |
| Initial Probability | π | P(starting in each state) |


In [ ]:
# Hidden states
states     = ['Bull', 'Bear', 'Neutral']
state_map  = {'Bull': 0, 'Bear': 1, 'Neutral': 2}

# Transition Matrix A
# A[i][j] = P(next_state=j | current_state=i)
A = [
    [0.6, 0.2, 0.2],   # Bull   → Bull, Bear, Neutral
    [0.2, 0.5, 0.3],   # Bear   → Bull, Bear, Neutral
    [0.3, 0.3, 0.4]    # Neutral→ Bull, Bear, Neutral
]

# Emission Matrix B
# B[i][k] = P(observation=k | state=i)
# Observations: Up=0, Down=1, Stable=2
B = [
    [0.7, 0.1, 0.2],   # Bull   → Up, Down, Stable
    [0.1, 0.7, 0.2],   # Bear   → Up, Down, Stable
    [0.3, 0.3, 0.4]    # Neutral→ Up, Down, Stable
]

# Initial probability π
pi = [0.4, 0.3, 0.3]  # Bull, Bear, Neutral

print('HMM Parameters defined!')
print(f'States      : {states}')
print(f'Observations: Up, Down, Stable')
print(f'Initial π   : {pi}')

# Display as DataFrames
print('\nTransition Matrix A:')
print(pd.DataFrame(A, index=states, columns=states).round(2))
print('\nEmission Matrix B:')
print(pd.DataFrame(B, index=states, columns=['Up','Down','Stable']).round(2))


## Step 9: Forward Algorithm

Calculates **P(observation sequence | model)** — probability that the model generated the observed sequence.

**Formula:**
- Init: α₀(i) = π(i) × B(i, O₀)
- Recurse: αₜ(j) = [Σᵢ αₜ₋₁(i) × A(i,j)] × B(j, Oₜ)
- Final: P(O|λ) = Σᵢ αₜ(i)


In [ ]:
alpha = []

# Initialization
first_obs = obs_map[obs_seq[0]]
alpha.append([pi[i] * B[i][first_obs] for i in range(3)])

print('=== Forward Algorithm ===')
print(f'Step 0 (obs={obs_seq[0]}): {[round(a,6) for a in alpha[0]]}')

# Recursion
for t in range(1, len(obs_seq)):
    obs_idx = obs_map[obs_seq[t]]
    temp    = []
    for j in range(3):
        sum_prob = sum(alpha[t-1][i] * A[i][j] for i in range(3))
        temp.append(sum_prob * B[j][obs_idx])
    alpha.append(temp)
    print(f'Step {t} (obs={obs_seq[t]}): {[round(a,6) for a in temp]}')

total_prob = sum(alpha[-1])
print(f'\nTotal Probability P(O|λ) = {total_prob:.8f}')
print('(Probability that this HMM model generated the observed sequence)')


## Step 10: Viterbi Algorithm

Finds the **most likely sequence of hidden states** given the observations.

**Formula:**
- Init: δ₀(i) = π(i) × B(i, O₀)
- Recurse: δₜ(j) = maxᵢ[δₜ₋₁(i) × A(i,j)] × B(j, Oₜ)
- Backtrack to find best path


In [ ]:
delta = []
psi   = []

# Initialization
delta.append([pi[i] * B[i][obs_map[obs_seq[0]]] for i in range(3)])
psi.append([0, 0, 0])

print('=== Viterbi Algorithm ===')
print(f'Step 0 (obs={obs_seq[0]}): {[round(d,6) for d in delta[0]]}')

# Recursion
for t in range(1, len(obs_seq)):
    obs_idx    = obs_map[obs_seq[t]]
    temp_delta = []
    temp_psi   = []
    for j in range(3):
        vals      = [delta[t-1][i] * A[i][j] for i in range(3)]
        max_val   = max(vals)
        max_state = vals.index(max_val)
        temp_delta.append(max_val * B[j][obs_idx])
        temp_psi.append(max_state)
    delta.append(temp_delta)
    psi.append(temp_psi)
    print(f'Step {t} (obs={obs_seq[t]}): {[round(d,6) for d in temp_delta]}')

# Backtracking
states_index = []
last_state   = int(np.argmax(delta[-1]))
states_index.append(last_state)

for t in range(len(obs_seq)-1, 0, -1):
    last_state = psi[t][last_state]
    states_index.insert(0, last_state)

result_states = [states[i] for i in states_index]

print('\n=== Most Likely Hidden State Sequence ===')
for t, (obs, state) in enumerate(zip(obs_seq, result_states)):
    print(f'  Step {t+1}: Observed={obs:6s} → Hidden State={state}')


## Step 11: Visualize HMM — Observations vs Hidden States

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Observations
obs_colors = {'Up': 'green', 'Down': 'red', 'Stable': 'gray'}
obs_vals   = [obs_map[o] for o in obs_seq]
colors_obs = [obs_colors[o] for o in obs_seq]

axes[0].bar(range(len(obs_seq)), obs_vals, color=colors_obs, edgecolor='black')
axes[0].set_title('Observed Sequence (Up=0, Down=1, Stable=2)', fontweight='bold')
axes[0].set_xticks(range(len(obs_seq)))
axes[0].set_xticklabels([f'T{i+1}\n{o}' for i, o in enumerate(obs_seq)])
axes[0].set_ylabel('Observation')
axes[0].grid(True, alpha=0.3)

# Hidden states
state_colors = {'Bull': 'green', 'Bear': 'red', 'Neutral': 'gray'}
state_vals   = [state_map[s] for s in result_states]
colors_state = [state_colors[s] for s in result_states]

axes[1].bar(range(len(result_states)), state_vals, color=colors_state, edgecolor='black')
axes[1].set_title('Most Likely Hidden States (Bull=0, Bear=1, Neutral=2)', fontweight='bold')
axes[1].set_xticks(range(len(result_states)))
axes[1].set_xticklabels([f'T{i+1}\n{s}' for i, s in enumerate(result_states)])
axes[1].set_ylabel('Hidden State')
axes[1].grid(True, alpha=0.3)

from matplotlib.patches import Patch
legend = [Patch(color='green', label='Up/Bull'),
          Patch(color='red',   label='Down/Bear'),
          Patch(color='gray',  label='Stable/Neutral')]
axes[1].legend(handles=legend, loc='upper right')

plt.suptitle('HMM: Observations vs Hidden States (Viterbi)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 12: HMM Future State Prediction

In [ ]:
future_days = int(input('\nEnter number of future days to predict using HMM: '))

current_state = states_index[-1]
future_states = []

print(f'\nStarting from last state: {states[current_state]}')
print('\nHMM Future Predictions:')

for i in range(future_days):
    probs      = A[current_state]
    next_state = np.random.choice([0, 1, 2], p=probs)
    future_states.append(states[next_state])
    print(f'  Day {i+1}: {states[next_state]}')
    current_state = next_state

# Visualize future states
state_num  = [state_map[s] for s in future_states]
bar_colors = [state_colors[s] for s in future_states]

plt.figure(figsize=(10, 4))
plt.bar(range(1, future_days+1), state_num, color=bar_colors, edgecolor='black')
plt.xticks(range(1, future_days+1), [f'Day {i+1}\n{s}' for i, s in enumerate(future_states)])
plt.title(f'HMM Future State Prediction ({future_days} days)', fontweight='bold')
plt.ylabel('Predicted State')
plt.yticks([0,1,2], ['Bull','Bear','Neutral'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
# Part 2B: Hidden Markov Model — Using `hmmlearn` Library
---

## Overview
While Part 2 implemented HMM **from scratch**, this section uses the **`hmmlearn`** library to:
- Train a Gaussian HMM on real AAPL price data (using returns)
- Decode hidden market states automatically
- Compare library results with manual HMM

| Aspect | Manual HMM (Part 2) | hmmlearn HMM (Part 2B) |
|---|---|---|
| Input | Discrete (U/D/S) | Continuous (daily returns) |
| Parameters | Manually defined | Learned from data |
| Forward / Viterbi | Hand-coded | Built-in |
| Use case | Education / Interpretable | Real-world / Data-driven |


## Step 12B-1: Install & Import `hmmlearn`

In [ ]:
# Install hmmlearn (if not already installed)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'hmmlearn', '-q'], check=True)

from hmmlearn.hmm import GaussianHMM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('hmmlearn imported successfully!')


## Step 12B-2: Prepare Features for hmmlearn

We use **daily returns** and **log returns** as observation features.  
hmmlearn fits a Gaussian distribution per hidden state to continuous data.


In [ ]:
# Compute daily returns from full AAPL dataset
df['Returns']     = df['Price'].pct_change()            # percentage return
df['Log_Returns'] = np.log(df['Price'] / df['Price'].shift(1))  # log return

# Drop NaN from first row
df_hmm = df[['Returns', 'Log_Returns']].dropna().copy()

print(f'Feature matrix shape: {df_hmm.shape}')
print(df_hmm.head())

# Prepare as 2D numpy array (required by hmmlearn)
X = df_hmm.values
lengths = [len(X)]   # single continuous sequence

print(f'\nX shape: {X.shape}')
print('Features: [Returns, Log_Returns]')


## Step 12B-3: Fit Gaussian HMM

We fit a **3-state Gaussian HMM** (Bull / Bear / Neutral) on the daily returns.  
hmmlearn uses **Baum-Welch (EM)** algorithm to learn:
- Transition matrix A
- Emission means & covariances
- Initial state probabilities π


In [ ]:
np.random.seed(42)

# Fit 3-state Gaussian HMM
hmm_model = GaussianHMM(
    n_components=3,       # 3 hidden states
    covariance_type='full',
    n_iter=200,
    random_state=42,
    verbose=False
)

hmm_model.fit(X, lengths)

print('=== GaussianHMM Fitted Successfully! ===')
print(f'Converged     : {hmm_model.monitor_.converged}')
print(f'Iterations    : {hmm_model.monitor_.iter}')
print(f'Log-likelihood: {hmm_model.score(X):.4f}')

print('\nLearned Transition Matrix A:')
print(pd.DataFrame(
    hmm_model.transmat_.round(3),
    index=[f'State {i}' for i in range(3)],
    columns=[f'State {i}' for i in range(3)]
))

print('\nLearned Emission Means (Returns, Log_Returns):')
print(pd.DataFrame(
    hmm_model.means_.round(6),
    columns=['Mean Return', 'Mean Log-Return'],
    index=[f'State {i}' for i in range(3)]
))


## Step 12B-4: Decode Hidden States (Viterbi via hmmlearn)

`hmm_model.predict()` runs the **Viterbi algorithm** internally to find  
the most likely hidden state sequence for each time step.


In [ ]:
# Decode hidden states using built-in Viterbi
hidden_states = hmm_model.predict(X)

# Label states by their mean return (highest return = Bull, lowest = Bear)
state_means = hmm_model.means_[:, 0]   # use Returns column
sorted_idx  = np.argsort(state_means)  # ascending: Bear, Neutral, Bull

label_map = {}
label_map[sorted_idx[0]] = 'Bear'
label_map[sorted_idx[1]] = 'Neutral'
label_map[sorted_idx[2]] = 'Bull'

state_labels = [label_map[s] for s in hidden_states]

df_hmm['Hidden_State'] = state_labels
df_hmm['Hidden_State_Num'] = hidden_states

print('Hidden State Distribution:')
print(df_hmm['Hidden_State'].value_counts())
print(f'\nFirst 10 decoded states:')
print(df_hmm['Hidden_State'].head(10).values)


## Step 12B-5: Visualize Hidden States on Price Chart

In [ ]:
# Align price data with decoded states
df_viz = df.loc[df_hmm.index].copy()
df_viz['Hidden_State'] = state_labels

state_colors_lib = {'Bull': 'green', 'Bear': 'red', 'Neutral': 'gray'}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ─── Top: Price colored by regime ───
for state, color in state_colors_lib.items():
    mask = df_viz['Hidden_State'] == state
    axes[0].scatter(df_viz.index[mask], df_viz['Price'][mask],
                    c=color, s=3, alpha=0.7, label=state)

axes[0].set_title('AAPL Price Colored by hmmlearn Hidden State', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(markerscale=5)
axes[0].grid(True, alpha=0.3)

# ─── Bottom: Hidden state over time ───
state_num_map = {'Bull': 2, 'Neutral': 1, 'Bear': 0}
state_nums    = [state_num_map[s] for s in state_labels]
colors_bar    = [state_colors_lib[s] for s in state_labels]

axes[1].bar(range(len(state_nums)), state_nums, color=colors_bar, width=1.0, alpha=0.8)
axes[1].set_title('Hidden State Sequence Decoded by hmmlearn (Viterbi)', fontweight='bold')
axes[1].set_ylabel('State')
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['Bear', 'Neutral', 'Bull'])
axes[1].set_xlabel('Time Steps')
axes[1].grid(True, alpha=0.3)

legend_patches = [mpatches.Patch(color='green', label='Bull'),
                  mpatches.Patch(color='gray',  label='Neutral'),
                  mpatches.Patch(color='red',   label='Bear')]
axes[1].legend(handles=legend_patches)

plt.suptitle('hmmlearn: Market Regime Detection on AAPL', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 12B-6: Posterior State Probabilities

`hmm_model.predict_proba()` returns the **posterior probability** of being in each  
state at each time step — giving a "soft" view of regime uncertainty.


In [ ]:
# Get posterior probabilities
posterior = hmm_model.predict_proba(X)

# Map to named states
col_names = [label_map[i] for i in range(3)]
df_posterior = pd.DataFrame(posterior, columns=col_names, index=df_hmm.index)

# Plot last 200 days for clarity
df_post_tail = df_posterior.tail(200)

fig, ax = plt.subplots(figsize=(14, 5))
ax.stackplot(df_post_tail.index,
             df_post_tail['Bull'].values,
             df_post_tail['Neutral'].values,
             df_post_tail['Bear'].values,
             labels=['Bull', 'Neutral', 'Bear'],
             colors=['green', 'gray', 'red'],
             alpha=0.6)

ax.set_title('Posterior State Probabilities (Last 200 Days)', fontsize=13, fontweight='bold')
ax.set_ylabel('Probability')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nMean posterior probabilities (overall):')
print(df_posterior.mean().round(4))


## Step 12B-7: Future State Prediction using hmmlearn Transition Matrix

In [ ]:
future_days_lib = int(input('\nEnter number of future days to predict using hmmlearn HMM: '))

# Start from the last decoded state
last_state_idx  = hidden_states[-1]
current_probs   = np.zeros(3)
current_probs[last_state_idx] = 1.0

print(f'Starting from: {label_map[last_state_idx]} (State {last_state_idx})')
print('\nhmmlearn Future State Predictions:')

future_preds = []
A_lib = hmm_model.transmat_

for day in range(future_days_lib):
    next_probs  = current_probs @ A_lib
    next_state  = np.argmax(next_probs)
    next_label  = label_map[next_state]
    future_preds.append((day+1, next_label, next_probs.copy()))
    print(f'  Day {day+1}: {next_label:<8} | Probs → Bull:{next_probs[sorted_idx[2]]:.3f}  Neutral:{next_probs[sorted_idx[1]]:.3f}  Bear:{next_probs[sorted_idx[0]]:.3f}')
    current_probs = next_probs

# Visualize
days_   = [p[0] for p in future_preds]
labels_ = [p[1] for p in future_preds]
nums_   = [state_num_map[p[1]] for p in future_preds]
clrs_   = [state_colors_lib[p[1]] for p in future_preds]

plt.figure(figsize=(10, 4))
plt.bar(days_, nums_, color=clrs_, edgecolor='black')
plt.xticks(days_, [f'Day {d}\n{l}' for d, l in zip(days_, labels_)], fontsize=8)
plt.yticks([0, 1, 2], ['Bear', 'Neutral', 'Bull'])
plt.title(f'hmmlearn Future State Prediction ({future_days_lib} days)', fontweight='bold')
plt.ylabel('Predicted State')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Step 12B-8: Manual HMM vs hmmlearn Comparison

| Feature | Manual HMM (Part 2) | hmmlearn (Part 2B) |
|---|---|---|
| Input type | Discrete (U / D / S) | Continuous (daily returns) |
| Parameters | Hand-defined by user | Learned via Baum-Welch (EM) |
| Forward Algorithm | Coded from scratch | Built-in `.score()` |
| Viterbi | Coded from scratch | Built-in `.predict()` |
| Posterior probs | Not computed | `.predict_proba()` |
| Scalability | Small sequences | Full historical data (~2000 pts) |
| Interpretability | High (you set A, B, π) | Lower (parameters are learned) |
| Real-world use | Teaching / demos | Production / research |

**Key takeaway:** Manual HMM builds intuition; hmmlearn scales to real data and learns parameters automatically.


---
# Part 3: ARIMA Implementation
---

## Overview
**ARIMA(p, d, q):**
| Parameter | Meaning | Determined by |
|---|---|---|
| p | AR order — number of past values used | PACF plot |
| d | Differencing order — to make series stationary | ADF test |
| q | MA order — number of past errors used | ACF plot |

From our analysis: **d=1** (one differencing needed), **p=1, q=1** (from ACF/PACF)


## Step 13: Prepare Data & Train/Test Split

In [ ]:
# Use last 500 days for faster training
series = df['Price'].tail(500)

# 80/20 split
split  = int(len(series) * 0.8)
train  = series[:split]
test   = series[split:]

print(f'Total data points : {len(series)}')
print(f'Training set      : {len(train)} points')
print(f'Test set          : {len(test)} points')

plt.figure(figsize=(12, 4))
plt.plot(train, color='steelblue', label='Train')
plt.plot(test,  color='orange',    label='Test')
plt.title('AAPL Price — Train / Test Split', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Step 14: Fit ARIMA Model

In [ ]:
print('Fitting ARIMA(1, 1, 1) model...')

# Fit ARIMA
arima_model = ARIMA(train, order=(1, 1, 1))
arima_fit   = arima_model.fit()

print('Model fitted successfully!')
print('\nModel Summary:')
print(arima_fit.summary())


## Step 15: Forecast & Evaluate

In [ ]:
# Forecast on test set
forecast_steps = len(test)
forecast       = arima_fit.forecast(steps=forecast_steps)
forecast_index = test.index

# Metrics
rmse = np.sqrt(mean_squared_error(test, forecast))
mae  = mean_absolute_error(test, forecast)

print('=== ARIMA Forecast Results ===')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')

# Plot actual vs predicted
plt.figure(figsize=(12, 5))
plt.plot(train[-50:],            color='steelblue', linewidth=1.5, label='Train (last 50)')
plt.plot(test,                   color='orange',    linewidth=1.5, label='Actual')
plt.plot(forecast_index, forecast, color='red',    linewidth=1.5, label='ARIMA Forecast', linestyle='--')
plt.title('ARIMA(1,1,1) — Actual vs Forecast', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Step 16: Future Price Forecast

In [ ]:
future_days_arima = int(input('Enter number of future days to forecast using ARIMA: '))

# Refit on full series for future forecast
arima_full    = ARIMA(series, order=(1, 1, 1)).fit()
future_fc     = arima_full.forecast(steps=future_days_arima)

# Future dates
last_date     = series.index[-1]
future_dates  = pd.date_range(start=last_date, periods=future_days_arima+1, freq='B')[1:]

print(f'\nARIMA Future Forecast ({future_days_arima} days):')
for i, (date, val) in enumerate(zip(future_dates, future_fc)):
    print(f'  Day {i+1} ({date.date()}): ${val:.2f}')

# Plot
plt.figure(figsize=(12, 5))
plt.plot(series[-60:],            color='steelblue', linewidth=1.5, label='Historical (last 60d)')
plt.plot(future_dates, future_fc, color='red',       linewidth=2,   label='Future Forecast', linestyle='--', marker='o', markersize=5)
plt.title(f'ARIMA — Future Price Forecast ({future_days_arima} days)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
# Part 4: Comparison — Manual HMM vs hmmlearn HMM vs ARIMA
---


## Step 17: Side-by-Side Comparison

In [ ]:
print('=' * 60)
print(f"{'HMM vs ARIMA Comparison':^60}")
print('=' * 60)
print(f"{'Feature':<25} {'HMM':<18} {'ARIMA'}")
print('-' * 60)
print(f"{'Type':<25} {'Probabilistic':<18} {'Statistical'}")
print(f"{'Input':<25} {'Discrete sequence':<18} {'Continuous prices'}")
print(f"{'Output':<25} {'Hidden states':<18} {'Numeric forecast'}")
print(f"{'Predicts':<25} {'Bull/Bear/Neutral':<18} {'Exact price'}")
print(f"{'Stationarity needed':<25} {'No':<18} {'Yes (differencing)'}")
print(f"{'Handles seasonality':<25} {'No':<18} {'Partial'}")
print(f"{'Best for':<25} {'Market regimes':<18} {'Short-term forecast'}")
print('=' * 60)

print(f'\nARIMA Metrics:')
print(f'  RMSE : {rmse:.4f}')
print(f'  MAE  : {mae:.4f}')

print('\nHMM Output: Qualitative (Bull/Bear/Neutral states)')
print('ARIMA Output: Quantitative (exact price values)')

print(f'\nhmmlearn HMM:')
print(f'  States decoded: {df_hmm["Hidden_State"].value_counts().to_dict()}')
print(f'  Log-likelihood: {hmm_model.score(X):.4f}')


## Applications

**HMM:**
- Stock market regime detection (bull/bear)
- Speech recognition
- Bioinformatics (gene sequencing)
- Weather pattern prediction

**ARIMA:**
- Stock price forecasting
- Sales demand forecasting
- Economic indicator prediction
- Energy consumption forecasting


## Conclusion

This assignment performed Time Series Analysis on AAPL stock prices using three approaches:

1. **Time Series Properties Analysis:** The AAPL stock price showed a clear upward trend over 2015–2023. The original series was non-stationary (ADF p > 0.05) but became stationary after first differencing. ACF and PACF plots confirmed ARIMA(1,1,1) as appropriate.

2. **Hidden Markov Model (HMM):** Manually implemented the Forward Algorithm to compute observation probability and the Viterbi Algorithm to find the most likely hidden state sequence (Bull/Bear/Neutral). HMM provides qualitative market regime insights.

3. **ARIMA(1,1,1):** Fitted on training data and forecasted test set prices. Provides quantitative numeric forecasts with measurable RMSE and MAE metrics.

**Key takeaway:** HMM and ARIMA complement each other — HMM identifies *what kind* of market we're in, while ARIMA predicts *exact future prices*. Together they provide a comprehensive time series analysis framework.
